In [13]:
import pandas as pd
import mlflow
import dagshub
import mlflow.sklearn
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error 
from sklearn.ensemble import RandomForestRegressor  
from datetime import datetime    

In [15]:
# -------------------------------------
# STEP 1: Load dataset
# ------------------------------------- 
df1 = pd.read_csv(r"D:\All_Final_Ml_Models_Model - V2\Battery_temperature_predication_end_to_end\data\raw\2.3kWH_new_Battery_data.csv")
print("✅ Data loaded:", df1.shape) 

✅ Data loaded: (26376, 35)


In [16]:
# ------------------------------------------
# STEP 2: Define input and target columns
# ------------------------------------------
input_columns = (
    ['SOC', 'SOH', 'Battery_pack_total_voltage', 'Battery_current'] +    
    [f'Battery_{i}_Volt' for i in range(1, 17)] +
    [f'temperature_{i}' for i in range(1, 7)]
)
target_columns = [f'temperature_{i}' for i in range(1, 7) if i != 2] 

df1 = df1[input_columns].dropna().reset_index(drop=True)
#df1 = df1.head(25000)

In [17]:
# -----------------------------------
# STEP 3: FIFO sliding window dataset
# -----------------------------------   
def create_sliding_windows(data, target_cols, W=30, H=30):
    """ 
    Create sliding windows for time series forecasting.
    W: window size (past timesteps)
    H: horizon (steps ahead to predict)  
    """
    X, y = [], [] 
    target_idx = [data.columns.get_loc(c) for c in target_cols] 

    # Move forward by 1 row each time (FIFO deque style)
    for start in range(len(data) - W - H + 1): 
        end = start + W                # last index of window
        pred_idx = end + H - 1         # index of prediction point

        # Past W timesteps as input
        X.append(data.iloc[start:end, :].values)

        # Target values at prediction point
        y.append(data.iloc[pred_idx, target_idx].values)

    return np.array(X), np.array(y) 

In [18]:
W = 30  # window size
H = 30  # horizon
X, y = create_sliding_windows(df1, target_columns, W=W, H=H) 

print("📦 X shape:", X.shape)  # (samples, W, features)
print("🎯 y shape:", y.shape)  # (samples, num_targets) 

📦 X shape: (26317, 30, 26)
🎯 y shape: (26317, 5)


In [19]:
# ----------------------------------------
# STEP 4: Time-based split (no shuffling)
# ----------------------------------------
split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]   

from sklearn.model_selection import train_test_split 

#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) 

# Flatten windows for RandomForest
X_train_flat = X_train.reshape(X_train.shape[0], -1) 
X_test_flat = X_test.reshape(X_test.shape[0], -1)

In [ ]:

import dagshub

mlflow.set_tracking_uri('https://dagshub.com/NaveenKotyale3/Battery_Temperatre_Predication.mlflow')
dagshub.init(repo_owner='NaveenKotyale3', repo_name='Battery_Temperatre_Predication', mlflow=True)

# mlflow.set_experiment("Logistic Regression Baseline")
mlflow.set_experiment("Random_Forest_Regressor")

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

d:\All_Final_Ml_Models_Model - 
V2\Battery_temperature_predication_end_to_end\myenv\lib\site-packages\rich\live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=f7663d05-2b6c-40f4-8126-28d1e60639c4&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=4a895a838c1b3cdda0f942d6a85c82c99eeefd6a3d2fb9cddb88742ac9a8fb9c




Accessing as NaveenKotyale3

Initialized MLflow to track repo "NaveenKotyale3/Battery_Temperatre_Predication"

Repository NaveenKotyale3/Battery_Temperatre_Predication initialized!

2026/02/05 15:37:23 INFO mlflow.tracking.fluent: Experiment with name 'Random_Forest_Regressor' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/a3560687d18c483692afd21b7811ba4b', creation_time=1770286044730, experiment_id='0', last_update_time=1770286044730, lifecycle_stage='active', name='Random_Forest_Regressor', tags={}>

In [22]:
with mlflow.start_run():

    try:

        print("converting into standard scaler")

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_flat)
        X_test_scaled = scaler.transform(X_test_flat)

        print("Model training has started")
        rf_model = RandomForestRegressor(n_estimators=150, max_depth=20, random_state=42,n_jobs=-1) 
        rf_model.fit(X_train_scaled, y_train)
        print("Model traing has completed")

        mlflow.log_param("without random train test split")
        mlflow.log_param("model","random forest")
        mlflow.log_param("n_estimators",150)
        mlflow.log_param("max_depth",20)

        y_pred_rf = rf_model.predict(X_test_scaled)

        print("calculating the evaluation metrics")

        r2_score = r2_score(y_test,y_pred_rf)
        mae = mean_absolute_error(y_test,y_pred_rf)
        rmse = np.sqrt(mean_squared_error(y_test,y_pred_rf))


        mlflow.log_metric("r2_score",r2_score)
        mlflow.log_metric("mae",mae)
        mlflow.log_metric("rmse",rmse)


        print("saving the model")
        mlflow.sklearn.log_model(rf_model,"random_forest_model")

    except Exception as e:
        print("An error has Occured")
    
    
    




converting into standard scaler
Model training has started
Model traing has completed
An error has Occured
🏃 View run inquisitive-vole-985 at: https://dagshub.com/NaveenKotyale3/Battery_Temperatre_Predication.mlflow/#/experiments/0/runs/2b9788f0f7324dfab7c0721aa05eff9f
🧪 View experiment at: https://dagshub.com/NaveenKotyale3/Battery_Temperatre_Predication.mlflow/#/experiments/0
